# Day 2: Independent Lab — Build Your Own Extractor

## Overview

In this lab, you'll build a complete **extraction + evaluation pipeline** for a business domain of your choice.

## Deliverables

1. **Pydantic schema** defining your extraction structure
2. **Extractor prompt v1** (baseline)
3. **Extractor prompt v2** (improved based on evaluation)
4. **Golden set** of 8+ labeled items
5. **Metrics** comparing v1 vs v2 accuracy
6. **Error analysis** documenting what you learned

## Choose Your Track

- **Track A:** Customer Support Triage (tickets → category, urgency, action)
- **Track B:** Meeting Notes Extraction (notes → attendees, actions, decisions)
- **Track C:** Product Review Analysis (reviews → sentiment, features, intent)
- **Track D:** Job Posting Parser (postings → requirements, benefits, red flags)

## Time Estimate: 60-90 minutes

---

## Setup

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os
import time
import json
import pandas as pd
from datetime import datetime, timezone
from typing import List, Optional, Literal
from pydantic import BaseModel, Field
from google import genai
from google.genai import types

# Configure API — reads the GEMINI_API_KEY you set up on Day 1
try:
    from google.colab import userdata
    API_KEY = userdata.get("GEMINI_API_KEY")
except:
    API_KEY = None

if not API_KEY:
    import getpass
    API_KEY = getpass.getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=API_KEY)
MODEL_ID = "gemini-2.5-flash-lite"

print(f"✓ API key loaded")
print(f"✓ Model: {MODEL_ID}")

In [ ]:
# Infrastructure
PROMPT_LOG = []

def _now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

def generate_structured(prompt, schema_model, temperature=0.2, log=True, label=None):
    """
    Generate structured output using Pydantic schema.
    The API guarantees the output matches your schema.
    """
    start_time = time.time()
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "temperature": temperature,
            "response_mime_type": "application/json",
            "response_json_schema": schema_model.model_json_schema(),
        },
    )
    
    latency = time.time() - start_time
    raw_text = response.text or ""
    result = schema_model.model_validate_json(raw_text)
    
    if log:
        PROMPT_LOG.append({
            "timestamp": _now(),
            "label": label,
            "schema": schema_model.__name__,
            "prompt": prompt[:300] + "..." if len(prompt) > 300 else prompt,
            "prompt_length": len(prompt),
            "response": raw_text[:300] + "..." if len(raw_text) > 300 else raw_text,
            "response_length": len(raw_text),
            "latency_s": round(latency, 3)
        })
    
    return result

def batch(items, batch_size=6):
    """Split items into batches."""
    for i in range(0, len(items), batch_size):
        yield items[i:i+batch_size]

def evaluate(predictions, golden, field):
    """Calculate accuracy for a field against golden set."""
    correct = 0
    total = 0
    errors = []
    
    for id, expected in golden.items():
        if id in predictions:
            total += 1
            pred_value = getattr(predictions[id], field)
            if pred_value == expected[field]:
                correct += 1
            else:
                errors.append({"id": id, "expected": expected[field], "got": pred_value})
    
    return {
        "correct": correct,
        "total": total,
        "accuracy": correct / total if total > 0 else 0,
        "errors": errors
    }

def compare_versions(v1_preds, v2_preds, golden, fields):
    """Compare two versions across multiple fields."""
    results = []
    for field in fields:
        v1_eval = evaluate(v1_preds, golden, field)
        v2_eval = evaluate(v2_preds, golden, field)
        results.append({
            "field": field,
            "v1_accuracy": f"{v1_eval['accuracy']:.1%}",
            "v2_accuracy": f"{v2_eval['accuracy']:.1%}",
            "improvement": f"{v2_eval['accuracy'] - v1_eval['accuracy']:+.1%}"
        })
    return pd.DataFrame(results)

print("✓ Infrastructure ready")

---

## Step 1: Choose Your Track and Define Inputs

Select your track and either use the provided sample data or create your own.

In [ ]:
# Mark your track
CHOSEN_TRACK = ""  # Fill in: "A", "B", "C", or "D"

print(f"Selected track: {CHOSEN_TRACK}")

In [ ]:
# Sample data for Track A: Customer Support
TRACK_A_DATA = [
    {"id": "T1", "text": "App crashes when uploading photos. Tried reinstalling, still broken."},
    {"id": "T2", "text": "Can you add dark mode? Would really help with eye strain at night."},
    {"id": "T3", "text": "What's your refund policy for annual subscriptions?"},
    {"id": "T4", "text": "Your service has been down for 3 hours! I'm losing business!"},
    {"id": "T5", "text": "How do I export my data to CSV format?"},
    {"id": "T6", "text": "I was promised a discount but charged full price."},
    {"id": "T7", "text": "The mobile app is much slower than the website."},
    {"id": "T8", "text": "Can I upgrade from Basic to Pro mid-subscription?"},
    {"id": "T9", "text": "Dashboard shows wrong numbers. Revenue chart is definitely broken."},
    {"id": "T10", "text": "Love the product! Any plans for an API?"},
    {"id": "T11", "text": "My account was charged twice this month!"},
    {"id": "T12", "text": "Integration with Slack stopped working after your update."},
]

# Sample data for Track B: Meeting Notes
TRACK_B_DATA = [
    {"id": "M1", "text": "Product sync - Jan 15. Attendees: Sarah (PM), Mike (Eng), Lisa (Design). Decided to prioritize mobile app over API. Lisa will deliver mockups by Jan 22. Next meeting: Jan 22, 2pm."},
    {"id": "M2", "text": "Quick sync with John and Amy about the customer complaint. Need to investigate database performance. John will look into it today. No decisions yet."},
    {"id": "M3", "text": "Q1 Planning. Team agreed on 3 OKRs. Budget approved for contractor hire. Sarah to draft job posting by Friday. Review in 2 weeks."},
    {"id": "M4", "text": "Design review with Lisa, Tom, and the CEO. Approved new logo concept. Tom to prepare brand guidelines. Launch planned for March."},
    {"id": "M5", "text": "1:1 with Alex. Discussed career growth and upcoming projects. Alex interested in leading the analytics initiative."},
    {"id": "M6", "text": "Incident postmortem. Root cause: misconfigured cache. Action items: add monitoring (DevOps, EOW), update runbook (SRE, next week), schedule training."},
    {"id": "M7", "text": "Sales pipeline review. 3 deals likely to close this quarter. Need demo environment fixed ASAP. Marketing to send case studies."},
    {"id": "M8", "text": "Brainstorm session for new features. No concrete decisions. Will reconvene after customer interviews."},
    {"id": "M9", "text": "Sprint retrospective with full dev team. Wins: shipped auth module on time. Struggles: too many context switches. Action: Mike to limit WIP to 2 items per person next sprint."},
    {"id": "M10", "text": "Client onboarding call with Acme Corp. Attendees: Rachel (CS), Dave (Solutions Eng), client reps. Agreed on 4-week rollout plan. Rachel to send onboarding checklist by Monday."},
    {"id": "M11", "text": "Weekly leadership sync — Feb 3. Headcount freeze confirmed through Q2. Decision: pause backfill for the open analyst role. CFO to share updated budget next week."},
    {"id": "M12", "text": "Security review meeting. Attendees: InfoSec team, two external auditors. Found 2 medium-severity vulnerabilities in payment flow. DevOps to patch by EOD Friday. Follow-up audit scheduled March 10."},
]

# Sample data for Track C: Product Review Analysis
TRACK_C_DATA = [
    {"id": "R1", "text": "Absolutely love this laptop! The screen is gorgeous and performance is blazing fast. Only complaint: the fan gets a bit loud under heavy load. Would definitely buy again."},
    {"id": "R2", "text": "Terrible experience. Battery died after 3 months and customer support was unhelpful. Returning it."},
    {"id": "R3", "text": "It's fine for the price. Camera is decent, build quality is okay. Nothing special but gets the job done."},
    {"id": "R4", "text": "The noise cancellation on these headphones is incredible, best I've ever used. But the ear cups get uncomfortable after about 2 hours. Sound quality is top-notch though."},
    {"id": "R5", "text": "DO NOT BUY. Arrived damaged, replacement also had issues. The software is buggy and crashes constantly. Complete waste of money."},
    {"id": "R6", "text": "Great value for a budget tablet. Kids love it for games and videos. Screen could be sharper but at this price point I'm not complaining."},
    {"id": "R7", "text": "Upgraded from the previous model. The new camera system is a real step up and the battery easily lasts all day. Face unlock is lightning fast. Best phone I've owned."},
    {"id": "R8", "text": "Mixed feelings. The design is beautiful and it looks premium on my desk. But the keyboard is mushy and the trackpad is too small. Productivity suffers."},
    {"id": "R9", "text": "Bought this for my home office. Setup was painless. Print quality is sharp for documents but photo printing is mediocre. Decent for the price."},
    {"id": "R10", "text": "Six months in and I regret this purchase. The smart features barely work, the app is a mess, and it's slower than my old dumb TV. Wasted $800."},
    {"id": "R11", "text": "Perfect for running! Tracks GPS accurately, heart rate monitor is reliable, and it survived several rainstorms. The companion app needs work though."},
    {"id": "R12", "text": "Looks exactly like the pictures. Fabric quality is surprisingly good for the price. Runs a bit small — order one size up. Would recommend to friends."},
]

# Sample data for Track D: Job Posting Parser
TRACK_D_DATA = [
    {"id": "J1", "text": "Senior Software Engineer - Remote. 5+ years experience in Python and cloud infrastructure. Competitive salary $150-180k. Must be comfortable with on-call rotations. Unlimited PTO."},
    {"id": "J2", "text": "Marketing Coordinator (Entry Level) - NYC Office. 0-2 years experience. Social media management, content creation. $45-55k. Great culture and mentorship program!"},
    {"id": "J3", "text": "VP of Engineering. Lead a team of 50+ engineers. 15+ years experience required. Must relocate to SF. Equity package available. We work hard and play hard."},
    {"id": "J4", "text": "Data Analyst - Hybrid (3 days in office). SQL, Python, Tableau required. 2-4 years experience. $75-95k + bonus. Family-friendly workplace with flexible hours."},
    {"id": "J5", "text": "Full Stack Developer - URGENT HIRE. Must start immediately. Competitive pay DOE. Fast-paced startup environment. Wear many hats. Rockstar developers only."},
    {"id": "J6", "text": "Product Manager - Remote OK. 3-5 years PM experience, preferably B2B SaaS. Strong analytical skills. $120-150k. Transparent salary bands, 4-day work week trial."},
    {"id": "J7", "text": "Junior UX Designer - Onsite, Austin TX. Portfolio required. Figma proficiency a must. $55-70k. Collaborative team, regular design critiques, conference budget."},
    {"id": "J8", "text": "DevOps Lead - Hybrid. 8+ years experience. Kubernetes, Terraform, AWS. Manage team of 4. Must be available 24/7 for critical incidents. Salary not listed."},
    {"id": "J9", "text": "Customer Success Manager - Remote. 2+ years in SaaS CSM role. Manage portfolio of enterprise accounts. $80-100k + commission. Clear promotion path to Senior CSM."},
    {"id": "J10", "text": "Machine Learning Engineer - Onsite, Seattle. PhD preferred. PyTorch, distributed training. Groundbreaking AI research. Publish papers. $180-250k + RSUs."},
    {"id": "J11", "text": "Executive Assistant to CEO - Onsite NYC. 5+ years supporting C-level. 60+ hour weeks expected. Discretion essential. Salary commensurate with experience."},
    {"id": "J12", "text": "Mid-Level Backend Engineer - Remote-first. Go or Rust experience. 3-5 years. $110-140k, equity, full benefits. Async communication, no meetings Wednesdays."},
]

In [ ]:
# Select your input data based on track
# Modify this cell based on your chosen track

inputs = []  # TODO: Set this to your track's data or your own data

# Example:
# inputs = TRACK_A_DATA
# inputs = TRACK_B_DATA
# inputs = TRACK_C_DATA
# inputs = TRACK_D_DATA
# OR define your own:
# inputs = [{"id": "X1", "text": "..."}, ...]

print(f"Input items: {len(inputs)}")

---

## Step 2: Define Your Schema

Create a Pydantic schema that captures the key information you want to extract.

**Requirements:**
- At least one `Literal` field for classification
- An urgency/priority field
- A summary field
- At least one action/recommendation field

In [ ]:
# TODO: Define your schema
# This is an example for Track A - modify for your track!

class ExtractedItem(BaseModel):
    """Extracted information from a single item."""
    id: str = Field(description="ID of the input item")
    
    # TODO: Add your classification field with Literal type
    # Example:
    # category: Literal["Bug", "Request", "Policy", "Complaint", "Other"]
    
    # TODO: Add urgency/priority field
    # Example:
    # urgency: Literal["low", "medium", "high"]
    
    # TODO: Add summary field
    # Example:
    # summary: str = Field(description="One-sentence summary (max 20 words)")
    
    # TODO: Add action/recommendation field
    # Example:
    # next_step: str = Field(description="Recommended action")
    
    # Optional: Add any additional fields relevant to your track
    # missing_info: List[str] = Field(description="Information needed to proceed")
    pass  # Remove this line when you add your fields


class ExtractedBatch(BaseModel):
    """Batch of extracted items."""
    items: List[ExtractedItem]

---

## Step 3: Write Prompt v1 (Baseline)

In [ ]:
# TODO: Write your v1 extraction prompt
# Keep it simple - this is your baseline

PROMPT_V1 = """
Extract information from each item into the specified schema.

Rules:
- Do NOT invent facts not present in the text
- If information is missing, note it appropriately
- Keep summaries concise

Items:
{items}
"""

def format_items(items):
    """Format items for the prompt."""
    return "\n".join([f"{it['id']}: {it['text']}" for it in items])

def run_extraction(prompt_template, items, label):
    """Run extraction with a prompt template."""
    prompt = prompt_template.format(items=format_items(items))
    return generate_structured(prompt, ExtractedBatch, label=label)

In [ ]:
# Run v1 extraction
# result_v1 = run_extraction(PROMPT_V1, inputs, "extraction_v1")
# print(f"Extracted: {len(result_v1.items)} items")

# View results
# df_v1 = pd.DataFrame([item.model_dump() for item in result_v1.items])
# df_v1

---

## Step 4: Create Your Golden Set (8+ items)

Manually label at least 8 items with the correct values for your classification and urgency fields.

**This is critical for evaluation!**

In [ ]:
# TODO: Create your golden set
# Label at least 8 items with ground truth

GOLDEN = {
    # Format: "id": {"field1": "value", "field2": "value"}
    # Example for Track A:
    # "T1": {"category": "Bug", "urgency": "high"},
    # "T2": {"category": "Request", "urgency": "low"},
    # ... at least 8 items
}

print(f"Golden set size: {len(GOLDEN)} items")
assert len(GOLDEN) >= 8, "Need at least 8 labeled items!"

---

## Step 5: Evaluate v1

In [ ]:
# Create predictions dict for evaluation
# pred_v1 = {item.id: item for item in result_v1.items}

# Evaluate - update field names to match your schema!
# cat_eval = evaluate(pred_v1, GOLDEN, "category")  # Change field name as needed
# urg_eval = evaluate(pred_v1, GOLDEN, "urgency")   # Change field name as needed

# print(f"V1 Category Accuracy: {cat_eval['correct']}/{cat_eval['total']} = {cat_eval['accuracy']:.1%}")
# print(f"V1 Urgency Accuracy:  {urg_eval['correct']}/{urg_eval['total']} = {urg_eval['accuracy']:.1%}")

# Show errors
# if cat_eval['errors']:
#     print("\nCategory errors:")
#     for err in cat_eval['errors']:
#         print(f"  {err['id']}: expected '{err['expected']}', got '{err['got']}'")

---

## Step 6: Improve Prompt (v2)

Based on your v1 errors, improve your prompt. Consider:
- Adding explicit decision rules for each category
- Adding examples (few-shot)
- Clarifying urgency definitions
- Adding edge case handling

In [ ]:
# TODO: Write your improved v2 prompt

PROMPT_V2 = PROMPT_V1 + """

# TODO: Add your improvements here
# Examples:

# CATEGORY RULES:
# - Bug: ...
# - Request: ...
# etc.

# URGENCY RULES:
# - high: ...
# - medium: ...
# - low: ...

# Or add few-shot examples:
# EXAMPLES:
# "App crashes" → Bug, high
# "Add dark mode" → Request, low
"""

In [ ]:
# Run v2 extraction
# result_v2 = run_extraction(PROMPT_V2, inputs, "extraction_v2")
# pred_v2 = {item.id: item for item in result_v2.items}

# Evaluate v2
# cat_eval_v2 = evaluate(pred_v2, GOLDEN, "category")
# urg_eval_v2 = evaluate(pred_v2, GOLDEN, "urgency")

# print(f"V2 Category Accuracy: {cat_eval_v2['correct']}/{cat_eval_v2['total']} = {cat_eval_v2['accuracy']:.1%}")
# print(f"V2 Urgency Accuracy:  {urg_eval_v2['correct']}/{urg_eval_v2['total']} = {urg_eval_v2['accuracy']:.1%}")

In [ ]:
# Compare versions
# comparison = compare_versions(pred_v1, pred_v2, GOLDEN, ["category", "urgency"])
# print("\n📊 Version Comparison:")
# print(comparison.to_string(index=False))

---

## Step 7: Error Analysis

Write your analysis in the markdown cell below.

### Error Analysis

#### Top 3 Error Patterns in v1

1. **[Error type 1]**: [Description and example]

2. **[Error type 2]**: [Description and example]

3. **[Error type 3]**: [Description and example]

#### Changes from v1 → v2

1. **[Change 1]**: [What you changed and why]

2. **[Change 2]**: [What you changed and why]

#### What You Would Try for v3

[Describe what improvements you would make with more time]

#### Remaining Risks

[What edge cases or risks remain? How would you mitigate them in production?]

---

## Step 8: Export Results

In [ ]:
# Export prompt log
if PROMPT_LOG:
    df_log = pd.DataFrame(PROMPT_LOG)
    df_log.to_csv("day2_independent_lab_log.csv", index=False)
    print(f"✓ Saved {len(PROMPT_LOG)} prompts to day2_independent_lab_log.csv")

# Export extracted results
# if result_v2:
#     with open("day2_extracted_results.json", "w") as f:
#         json.dump([item.model_dump() for item in result_v2.items], f, indent=2)
#     print("✓ Saved extracted results to day2_extracted_results.json")

---

## Checklist Before Proceeding

- [ ] Chose a track and defined input data (12+ items)
- [ ] Created Pydantic schema with Literal types
- [ ] Wrote baseline prompt (v1) and ran extraction
- [ ] Created golden set (8+ labeled items)
- [ ] Evaluated v1 accuracy
- [ ] Improved prompt (v2) with rules/examples
- [ ] Evaluated v2 and compared to v1
- [ ] Wrote error analysis
- [ ] Exported logs and results

---

**Proceed to: Day 2 Assignment →**